# RLWOA-SOFL paper adaptation: HAM10000 and blood cells

The source paper (Anter and Zhang, 2023, DOI 10.1109/TAFFC.2023.3285997) studies fMRI pain decoding. This notebook adapts its binary WOA feature selection, per-agent learned exploration, and self-organizing fuzzy prototypes to **two different binary datasets**. It does not reproduce the paper's clinical experiment or exact AnYa classifier. Attach Kaggle inputs `kmader/skin-cancer-mnist-ham10000` and `alitaqishah/blood-cell-anomaly-detection-2025`. No Fashion-MNIST is used.


In [ ]:
from pathlib import Path
import os, shutil
root = Path.cwd()
required = {'HAM10000_metadata.csv': root/'data'/'ham',
            'hmnist_8_8_RGB.csv': root/'data'/'ham',
            'blood_cell_anomaly_detection.csv': root/'data'/'blood'}
for name, directory in required.items():
    target = directory/name
    if not target.exists():
        matches = list(Path('/kaggle/input').rglob(name))
        if len(matches) != 1: raise FileNotFoundError(f'Expected one {name}; got {len(matches)}')
        directory.mkdir(parents=True, exist_ok=True)
        shutil.copy2(matches[0], target)
print('Data ready')


In [ ]:
import argparse, json, time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import expit, softmax
from sklearn.metrics import confusion_matrix, log_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split, StratifiedGroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
DATA = Path.cwd()/'data'
def load_dataset(name: str):
    if name == "HAM10000":
        pixels = pd.read_csv(DATA / "ham" / "hmnist_8_8_RGB.csv")
        meta = pd.read_csv(DATA / "ham" / "HAM10000_metadata.csv")
        mapping = {"akiec": 0, "bcc": 1, "bkl": 2, "df": 3,
                   "nv": 4, "vasc": 5, "mel": 6}
        if len(pixels) != len(meta) or not np.array_equal(
            pixels["label"].to_numpy(), meta["dx"].map(mapping).to_numpy()
        ):
            raise ValueError("HAM pixel/metadata row alignment failed")
        x = pixels.drop(columns="label").to_numpy(dtype=np.float32) / 255.0
        y = (meta["dx"] == "mel").astype(int).to_numpy()
        groups = meta["lesion_id"].astype(str).to_numpy()
        return x, y, groups, list(pixels.columns[:-1])
    if name == "Blood cell":
        df = pd.read_csv(DATA / "blood" / "blood_cell_anomaly_detection.csv")
        # Restrict to measured cell morphology, colour, and routine blood counts.
        # Exclude cell_type, disease_category, and three model/labeller scores.
        features = ["cell_diameter_um", "nucleus_area_pct", "chromatin_density",
                    "cytoplasm_ratio", "circularity", "eccentricity",
                    "granularity_score", "lobularity_score", "membrane_smoothness",
                    "cell_area_px", "perimeter_px", "mean_r", "mean_g", "mean_b",
                    "stain_intensity", "wbc_count_per_ul",
                    "rbc_count_millions_per_ul", "hemoglobin_g_dl",
                    "hematocrit_pct", "platelet_count_per_ul", "mcv_fl", "mchc_g_dl"]
        if df["cell_id"].duplicated().any() or df[features].isna().any().any():
            raise ValueError("Unexpected duplicate ID or missing blood feature")
        return df[features].to_numpy(dtype=np.float32), df["anomaly_label"].to_numpy(dtype=int), None, features
    raise ValueError(name)

def balanced_subsample(indices, labels, n, seed):
    if len(indices) <= n:
        return indices
    chosen, _ = train_test_split(indices, train_size=n, stratify=labels[indices], random_state=seed)
    return np.sort(chosen)

def split_data(x, y, groups, seed, train_n, test_n):
    idx = np.arange(len(y))
    if groups is None:
        tr, te = train_test_split(idx, test_size=0.28, stratify=y, random_state=seed)
    else:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.28, random_state=seed)
        tr, te = next(splitter.split(x, y, groups))
    tr = balanced_subsample(tr, y, train_n, seed)
    te = balanced_subsample(te, y, test_n, seed + 100)
    if groups is not None and set(groups[tr]) & set(groups[te]):
        raise AssertionError("Lesion leakage")
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        raise ValueError("Split has only one class")
    scaler = StandardScaler().fit(x[tr])
    return scaler.transform(x[tr]), scaler.transform(x[te]), y[tr], y[te], tr, te

def binary_metrics(y, prediction, probability):
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if precision + sensitivity else np.nan
    acc = (tp + tn) / len(y)
    prevalence = (tp + fn) / len(y)
    predicted = (tp + fp) / len(y)
    chance = prevalence * predicted + (1 - prevalence) * (1 - predicted)
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                accuracy=acc, error=1-acc, sensitivity=sensitivity,
                specificity=specificity, precision=precision, recall=sensitivity,
                f_measure=f1, auc_roc=roc_auc_score(y, probability),
                kappa=(acc-chance)/(1-chance),
                informedness=sensitivity+specificity-1,
                markedness=precision+npv-1,
                log_loss=log_loss(y, probability, labels=[0, 1]))

def evaluate(xtr, ytr, xte, yte, mask, seed):
    scaler = StandardScaler().fit(xtr[:, mask])
    clf = LogisticRegression(class_weight="balanced", max_iter=1000,
                             random_state=seed)
    clf.fit(scaler.transform(xtr[:, mask]), ytr)
    probability = clf.predict_proba(scaler.transform(xte[:, mask]))[:, 1]
    prediction = (probability >= .5).astype(int)
    return binary_metrics(yte, prediction, probability), probability, prediction

## Method

Five agents search for 20 iterations (105 fitness attempts). The paper's fitness is inner five-neighbor accuracy plus `alpha × (1 − selected/total)`. Standard WOA uses a linear exploration schedule; the adaptive method learns one three-action neural policy per agent from recent fitness changes. The fuzzy classifier updates class-specific cosine prototypes online. The training-only search and held-out test use lesion-separated splits for HAM10000 and stratified rows for blood cells. The fuzzy prototype implementation is an explicit approximation of the source paper's more elaborate AnYa rules.


In [ ]:
"""Auditable RLWOA-SOFL adaptation for the two AML benchmark tables.

This is an adaptation of Anter & Zhang (2023), DOI 10.1109/TAFFC.2023.3285997,
not a claim of exact reproduction: the source fMRI data and full SOFL reference
implementation are unavailable. Search fitness follows its KNN accuracy plus
feature sparsity equation; the learned action policy and fuzzy prototype model
are explicit, reproducible approximations documented in docs/PAPER_ADAPTATION.md.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit, softmax
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


ROOT = Path.cwd()


def repair(mask, rng):
    mask = np.asarray(mask, dtype=bool).copy()
    if mask.sum() < 2:
        mask[rng.choice(mask.size, 2, replace=False)] = True
    return mask


class PaperFitness:
    """Paper Eq. 14, evaluated on training-only inner folds."""

    def __init__(self, x, y, groups, seed, alpha=.8):
        self.x, self.y, self.alpha = x, y, alpha
        cv = (StratifiedKFold(2, shuffle=True, random_state=seed) if groups is None
              else StratifiedGroupKFold(2, shuffle=True, random_state=seed))
        self.folds = list(cv.split(x, y, groups))
        if groups is not None:
            assert all(set(groups[a]).isdisjoint(groups[b]) for a, b in self.folds)
        self.cache = {}
        self.attempts = 0

    def __call__(self, mask):
        self.attempts += 1
        key = mask.tobytes()
        if key not in self.cache:
            scores = []
            for tr, va in self.folds:
                scaler = StandardScaler().fit(self.x[tr][:, mask])
                model = KNeighborsClassifier(n_neighbors=5)
                model.fit(scaler.transform(self.x[tr][:, mask]), self.y[tr])
                scores.append(model.score(scaler.transform(self.x[va][:, mask]), self.y[va]))
            accuracy = float(np.mean(scores))
            self.cache[key] = (accuracy + self.alpha * (1 - mask.mean()), accuracy)
        return self.cache[key]


class AgentPolicy:
    """Small online neural state-action policy, trained from reward per agent."""

    def __init__(self, rng, hidden=7):
        self.w1 = rng.normal(0, .2, (3, hidden))
        self.b1 = np.zeros(hidden)
        self.w2 = rng.normal(0, .2, (hidden, 3))
        self.b2 = np.zeros(3)

    def choose(self, state, rng):
        h = np.tanh(state @ self.w1 + self.b1)
        p = softmax(h @ self.w2 + self.b2)
        return int(rng.choice(3, p=p)), h, p

    def learn(self, state, action, h, p, reward, rate=.035):
        # REINFORCE gradient on log policy; reward is sign of fitness change.
        delta = -p.copy()
        delta[action] += 1
        delta *= reward
        old_w2 = self.w2.copy()
        self.w2 += rate * np.outer(h, delta)
        self.b2 += rate * delta
        dh = (old_w2 @ delta) * (1 - h * h)
        self.w1 += rate * np.outer(state, dh)
        self.b1 += rate * dh


def optimize_woa(fitness, seed, adaptive, agents=5, iterations=20):
    rng = np.random.default_rng(seed)
    d = fitness.x.shape[1]
    positions = rng.random((agents, d))
    masks = np.vstack([repair(p >= .5, rng) for p in positions])
    values = np.array([fitness(m)[0] for m in masks])
    best_i = int(values.argmax())
    best, best_value = positions[best_i].copy(), float(values[best_i])
    best_mask = masks[best_i].copy()
    rates = np.full(agents, 2.0)
    policies = [AgentPolicy(rng) for _ in range(agents)]
    history = np.zeros((agents, 3))
    trace = [dict(iteration=0, best_fitness=best_value, selected=int(best_mask.sum()),
                  positive_rewards=0, negative_rewards=0)]
    positive = negative = 0
    for t in range(1, iterations + 1):
        next_positions = np.empty_like(positions)
        next_masks = np.empty_like(masks)
        next_values = np.empty_like(values)
        for i in range(agents):
            state = history[i].copy()
            if adaptive:
                action, hidden, probs = policies[i].choose(state, rng)
                rates[i] = np.clip(rates[i] * (1 + .1 * (action - 1)), .05, 2.0)
                a = rates[i]
            else:
                a = 2 * (1 - (t - 1) / iterations)
            r = rng.random(d)
            A = 2 * a * r - a
            C = 2 * rng.random(d)
            if rng.random() < .5:
                reference = positions[rng.integers(agents)] if np.mean(np.abs(A)) > 1 else best
                new_position = reference - A * np.abs(C * reference - positions[i])
            else:
                l = rng.uniform(-1, 1)
                new_position = np.abs(best - positions[i]) * np.exp(l) * np.cos(2 * np.pi * l) + best
            next_positions[i] = np.clip(new_position, 0, 1)
            # Paper uses a sigmoid transfer to map real coordinates to bits.
            next_masks[i] = repair(rng.random(d) < expit(4 * (next_positions[i] - .5)), rng)
            next_values[i] = fitness(next_masks[i])[0]
            reward = float(np.sign(next_values[i] - values[i]))
            positive += reward > 0
            negative += reward < 0
            history[i] = np.roll(history[i], -1)
            history[i, -1] = reward
            if adaptive:
                policies[i].learn(state, action, hidden, probs, reward)
        positions, masks, values = next_positions, next_masks, next_values
        j = int(values.argmax())
        if values[j] > best_value:
            best, best_mask, best_value = positions[j].copy(), masks[j].copy(), float(values[j])
        trace.append(dict(iteration=t, best_fitness=best_value,
                          selected=int(best_mask.sum()), positive_rewards=int(positive),
                          negative_rewards=int(negative)))
    assert fitness.attempts == agents * (iterations + 1)
    return best_mask, trace


class PrototypeFuzzyClassifier:
    """Online class prototypes with cosine similarity and soft fuzzy decisions.

    A tractable SOFL-inspired classifier, not the source paper's full AnYa model.
    """

    def __init__(self, radius=.25, max_prototypes=20, temperature=.12):
        self.radius = radius
        self.max_prototypes = max_prototypes
        self.temperature = temperature

    @staticmethod
    def _unit(x):
        norm = np.linalg.norm(x, axis=1, keepdims=True)
        return x / np.maximum(norm, 1e-12)

    def fit(self, x, y):
        x = self._unit(np.asarray(x, float))
        self.classes_ = np.unique(y)
        self.prototypes_ = {}
        for c in self.classes_:
            centers, counts = [], []
            for row in x[y == c]:
                if not centers:
                    centers.append(row.copy()); counts.append(1); continue
                sim = np.array(centers) @ row
                j = int(sim.argmax())
                if 1 - sim[j] > self.radius and len(centers) < self.max_prototypes:
                    centers.append(row.copy()); counts.append(1)
                else:
                    counts[j] += 1
                    centers[j] = (centers[j] * (counts[j] - 1) + row) / counts[j]
                    centers[j] /= max(np.linalg.norm(centers[j]), 1e-12)
            self.prototypes_[c] = np.array(centers)
        return self

    def predict_proba(self, x):
        x = self._unit(np.asarray(x, float))
        similarity = np.column_stack([np.max(x @ self.prototypes_[c].T, axis=1)
                                      for c in self.classes_])
        return softmax(similarity / self.temperature, axis=1)


def evaluate_fuzzy(xtr, ytr, xte, yte, mask, radius):
    scaler = StandardScaler().fit(xtr[:, mask])
    model = PrototypeFuzzyClassifier(radius=radius).fit(
        scaler.transform(xtr[:, mask]), ytr)
    prob = model.predict_proba(scaler.transform(xte[:, mask]))[:, 1]
    pred = (prob >= .5).astype(int)
    return binary_metrics(yte, pred, prob), prob, pred, sum(len(p) for p in model.prototypes_.values())


def run(args):
    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    runs, traces, predictions, selections = [], [], [], []
    for dataset in ("HAM10000", "Blood cell"):
        x, y, groups, feature_names = load_dataset(dataset)
        for seed in range(args.seeds):
            split_seed = 100 + seed
            _, _, ytr, yte, tr, te = split_data(x, y, groups, split_seed,
                                                args.train_n, args.test_n)
            xtr, xte = x[tr], x[te]
            for method in ("WOA + LR", "Adaptive WOA + LR", "Adaptive WOA + fuzzy", "Full + fuzzy"):
                start = time.process_time()
                with threadpool_limits(limits=1):
                    if method == "Full + fuzzy":
                        mask = np.ones(x.shape[1], bool)
                        trace = []
                        fit = None
                    else:
                        fit = PaperFitness(xtr, ytr, None if groups is None else groups[tr],
                                           split_seed, args.alpha)
                        mask, trace = optimize_woa(fit, 1000 + seed * 10 +
                                                   (0 if method == "WOA + LR" else 1),
                                                   method != "WOA + LR", args.agents, args.iterations)
                    if "fuzzy" in method:
                        metrics, prob, pred, n_prototypes = evaluate_fuzzy(
                            xtr, ytr, xte, yte, mask, args.radius)
                    else:
                        metrics, prob, pred = evaluate(xtr, ytr, xte, yte, mask, split_seed)
                        n_prototypes = 0
                cpu = time.process_time() - start
                runs.append(dict(dataset=dataset, seed=seed, method=method,
                                 alpha=args.alpha, radius=args.radius, n_train=len(tr),
                                 n_test=len(te), n_features=int(mask.sum()),
                                 n_prototypes=n_prototypes,
                                 evaluations=0 if fit is None else fit.attempts,
                                 inner_accuracy=np.nan if fit is None else fit.cache[mask.tobytes()][1],
                                 cpu_seconds=cpu, **metrics))
                selections.append(dict(dataset=dataset, seed=seed, method=method,
                                       features=json.dumps([feature_names[j] for j in np.flatnonzero(mask)])))
                traces.extend(dict(dataset=dataset, seed=seed, method=method, **point)
                              for point in trace)
                predictions.extend(dict(dataset=dataset, seed=seed, method=method,
                                        test_index=int(te[j]), y=int(yte[j]),
                                        probability=float(prob[j]), pred=int(pred[j]))
                                   for j in range(len(te)))
                print(f"{dataset} seed={seed} {method}: AUC={metrics['auc_roc']:.3f} "
                      f"features={mask.sum()} CPU={cpu:.2f}s", flush=True)
    df = pd.DataFrame(runs)
    df.to_csv(out / "runs.csv", index=False)
    pd.DataFrame(traces).to_csv(out / "trajectories.csv", index=False)
    pd.DataFrame(predictions).to_csv(out / "predictions.csv", index=False)
    pd.DataFrame(selections).to_csv(out / "selected_features.csv", index=False)
    df.groupby(["dataset", "method"], sort=False).agg(
        auc_roc_mean=("auc_roc", "mean"), auc_roc_sd=("auc_roc", "std"),
        accuracy_mean=("accuracy", "mean"), sensitivity_mean=("sensitivity", "mean"),
        specificity_mean=("specificity", "mean"), f_measure_mean=("f_measure", "mean"),
        log_loss_mean=("log_loss", "mean"), n_features_mean=("n_features", "mean"),
        cpu_seconds_mean=("cpu_seconds", "mean")).reset_index().to_csv(out / "summary.csv", index=False)
    (out / "protocol.json").write_text(json.dumps(vars(args), indent=2))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output", default=str(ROOT / "results" / "paper_rlwoa_sofl"))
    parser.add_argument("--seeds", type=int, default=3)
    parser.add_argument("--train-n", type=int, default=750)
    parser.add_argument("--test-n", type=int, default=300)
    parser.add_argument("--agents", type=int, default=5)
    parser.add_argument("--iterations", type=int, default=20)
    parser.add_argument("--alpha", type=float, default=.8)
    parser.add_argument("--radius", type=float, default=.25)
    args = parser.parse_args()
    run(args)




In [ ]:
from argparse import Namespace
for alpha, suffix in [(0.8, 'alpha08'), (0.2, 'alpha02')]:
    settings = Namespace(output=str(root/'aml-paper-adaptation'/suffix), seeds=3,
                         train_n=750, test_n=300, agents=5, iterations=20,
                         alpha=alpha, radius=.25)
    run(settings)


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, (alpha, suffix) in zip(axs, [(0.8,'alpha08'), (0.2,'alpha02')]):
    summary = pd.read_csv(root/'aml-paper-adaptation'/suffix/'summary.csv')
    display(summary.round(3))
    for dataset, marker in [('HAM10000','o'), ('Blood cell','s')]:
        sub = summary[summary.dataset == dataset]
        ax.plot(sub.n_features_mean, sub.auc_roc_mean, marker=marker, label=dataset)
    ax.set(title=f'alpha={alpha}', xlabel='Mean feature count', ylabel='Mean held-out ROC AUC')
    ax.grid(alpha=.2); ax.legend()
plt.show()


## Interpretation

Read ROC AUC together with log loss, confusion metrics, feature count, and process CPU time. The paper's `alpha=0.8` heavily favors sparse masks; this may remove useful blood features. The `alpha=0.2` run is a sensitivity analysis, not a test-selected winner. Three overlapping splits do not establish significance or patient-level generalization. FROC cannot be computed because neither table includes scored detection candidates and image locations.
